In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import h5py

subset_path = r"C:\Users\Ahmed\radioml_subset_100k.h5"

with h5py.File(subset_path, "r") as hf:
    X_small = hf["X"][:]
    Y_small = hf["Y"][:]
    Z_small = hf["Z"][:]

print(X_small.shape)
print(Y_small.shape)
print(Z_small.shape)

(100000, 1024, 2)
(100000, 24)
(100000, 1)


In [3]:
X_train, X_temp, Y_train, Y_temp, Z_train, Z_temp = train_test_split(
    X_small,
    Y_small,
    Z_small,
    test_size=0.3,
    random_state=42,
    stratify=np.argmax(Y_small, axis=1)
)

X_val, X_test, Y_val, Y_test, Z_val, Z_test = train_test_split(
    X_temp,
    Y_temp,
    Z_temp,
    test_size=0.5,
    random_state=42,
    stratify=np.argmax(Y_temp, axis=1)
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(70000, 1024, 2)
(15000, 1024, 2)
(15000, 1024, 2)


In [4]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(np.mean(X_train))
print(np.std(X_train))

7.193429e-09
1.0


In [5]:
np.save("X_test.npy", X_test)
np.save("Y_test.npy", Y_test)
np.save("X_val.npy", X_val)
np.save("Y_val.npy", Y_val)

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = (1024, 2)

model_k7 = models.Sequential([

    layers.Input(shape=input_shape),

    layers.Conv1D(
        64,
        kernel_size=7,
        activation='relu'
    ),
    layers.MaxPooling1D(2),

    layers.Conv1D(
        128,
        kernel_size=7,
        activation='relu'
    ),
    layers.MaxPooling1D(2),

    layers.Conv1D(
        256,
        kernel_size=7,
        activation='relu'
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(24, activation='softmax')
])

model_k7.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_k7.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 1018, 64)       │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 509, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 503, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 251, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 245, 256)       │       229,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 324,056 (1.24 MB)

 Trainable params: 324,056 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "improved_v2_kernel7.keras",
    monitor='val_loss',
    save_best_only=True
)

In [10]:
history_k7 = model_k7.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=40,
    batch_size=256,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 70s 251ms/step - accuracy: 0.1530 - loss: 2.6800 - val_accuracy: 0.2625 - val_loss: 2.2633
Epoch 2/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 72s 263ms/step - accuracy: 0.2715 - loss: 2.1678 - val_accuracy: 0.3161 - val_loss: 1.9758
Epoch 3/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 73s 266ms/step - accuracy: 0.3170 - loss: 1.9548 - val_accuracy: 0.3372 - val_loss: 1.8805
Epoch 4/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 72s 265ms/step - accuracy: 0.3406 - loss: 1.8612 - val_accuracy: 0.3581 - val_loss: 1.8081
Epoch 5/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 73s 267ms/step - accuracy: 0.3522 - loss: 1.8177 - val_accuracy: 0.3597 - val_loss: 1.7639
Epoch 6/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 72s 264ms/step - accuracy: 0.3617 - loss: 1.7884 - val_accuracy: 0.3657 - val_loss: 1.7540
Epoch 7/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 74s 269ms/step - accuracy: 0.3651 - loss: 1.7722 - val_accuracy: 0.3792 - val_loss: 1.7305
Epoch 8/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 73s 268ms/step - accuracy: 0.3717 - loss: 1

In [11]:
test_loss_k7, test_acc_k7 = model_k7.evaluate(
    X_test,
    Y_test,
    verbose=1
)

print("Kernel-7 Test Accuracy:", test_acc_k7)

469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4481 - loss: 1.5545
Kernel-7 Test Accuracy: 0.4480666518211365


In [12]:
model_k7.save("Kernel-7_model.keras")